https://drive.google.com/drive/folders/1p1JrxYtvBY2ciGjy1wrMH0CZSRBV1TLE

In [ ]:
!pip -q install pyspark

In [ ]:
from google.colab import files
from pyspark.sql import SparkSession

In [ ]:
from pyspark.sql.functions import col, sum, count, desc

In [ ]:
spark=SparkSession.builder.appName('Laazy_Loading_Tasks').getOrCreate()
uploaded=files.upload()

Saving lazy_execution_stores.csv to lazy_execution_stores.csv
Saving lazy_execution_products.csv to lazy_execution_products.csv
Saving lazy_execution_sales_25000.csv to lazy_execution_sales_25000.csv


In [ ]:
stores_df=spark.read.csv("lazy_execution_stores.csv",header=True,
    inferSchema=True)


In [ ]:
products_df=spark.read.csv("lazy_execution_products.csv",header=True,
    inferSchema=True)

In [ ]:
sales_df=spark.read.csv("lazy_execution_sales_25000.csv",header=True,
    inferSchema=True)

**Part A — Basic transformations and actions**

In [ ]:
df1=sales_df.filter(col("quantity")>5)

In [ ]:
df2=sales_df.select("order_id" , "product_id" , "quantity" , "unit_price")
df2.show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
| O000001|      P002|       3|      2999|
| O000002|      P014|       6|     29999|
| O000003|      P008|       6|     22999|
| O000004|      P008|       7|     22999|
| O000005|      P007|       2|     79999|
| O000006|      P005|       8|      2499|
| O000007|      P010|       8|      3999|
| O000008|      P002|       9|      2999|
| O000009|      P007|       5|     79999|
| O000010|      P006|       5|     55999|
| O000011|      P007|       1|     79999|
| O000012|      P010|       9|      3999|
| O000013|      P013|       6|      5999|
| O000014|      P005|       8|      2499|
| O000015|      P007|       7|     79999|
| O000016|      P006|       1|     55999|
| O000017|      P004|       9|     16999|
| O000018|      P012|       7|      6999|
| O000019|      P014|       5|     29999|
| O000020|      P010|       7|      3999|
+--------+----------+--------+----

In [ ]:
df3=sales_df.filter(col("unit_price") > 20000)
df3.count()

6560

In [ ]:
df4=sales_df.withColumn(
    "gross_amount",
    col("quantity") * col("unit_price")
)
df4.explain()
df4.show()

== Physical Plan ==
*(1) Project [order_id#60, order_date#61, store_id#62, product_id#63, quantity#64, unit_price#65, discount_pct#66, payment_method#67, order_status#68, sales_channel#69, (quantity#64 * unit_price#65) AS gross_amount#113]
+- FileScan csv [order_id#60,order_date#61,store_id#62,product_id#63,quantity#64,unit_price#65,discount_pct#66,payment_method#67,order_status#68,sales_channel#69] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...


+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|gross_amount|
+--------+----------+--------+----------+--------+-

In [ ]:
df5=sales_df.filter(col("order_status") == "Completed").orderBy(col("unit_price").desc())
df5.take(10)

[Row(order_id='O000085', order_date=datetime.date(2026, 4, 22), store_id='S03', product_id='P007', quantity=4, unit_price=79999, discount_pct=0, payment_method='Card', order_status='Completed', sales_channel='Online'),
 Row(order_id='O000290', order_date=datetime.date(2026, 6, 26), store_id='S06', product_id='P007', quantity=3, unit_price=79999, discount_pct=10, payment_method='Net Banking', order_status='Completed', sales_channel='Online'),
 Row(order_id='O000098', order_date=datetime.date(2026, 4, 15), store_id='S05', product_id='P007', quantity=1, unit_price=79999, discount_pct=5, payment_method='Net Banking', order_status='Completed', sales_channel='Online'),
 Row(order_id='O000005', order_date=datetime.date(2026, 3, 14), store_id='S06', product_id='P007', quantity=2, unit_price=79999, discount_pct=0, payment_method='Cash', order_status='Completed', sales_channel='Online'),
 Row(order_id='O000107', order_date=datetime.date(2026, 5, 24), store_id='S05', product_id='P007', quantity=9

**Part B — Chained transformations**

In [ ]:
df6=sales_df.filter(col("order_status") == "Completed")\
  .withColumn("gross_amount",col("quantity")*col("unit_price"))\
  .select( "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "gross_amount").sort(col("gross_amount").desc())
df6.show()

+--------+----------+--------+----------+------------+
|order_id|product_id|quantity|unit_price|gross_amount|
+--------+----------+--------+----------+------------+
| O002603|      P007|       9|     79999|      719991|
| O006170|      P007|       9|     79999|      719991|
| O002693|      P007|       9|     79999|      719991|
| O000107|      P007|       9|     79999|      719991|
| O002841|      P007|       9|     79999|      719991|
| O000367|      P007|       9|     79999|      719991|
| O002940|      P007|       9|     79999|      719991|
| O000527|      P007|       9|     79999|      719991|
| O003108|      P007|       9|     79999|      719991|
| O000690|      P007|       9|     79999|      719991|
| O003542|      P007|       9|     79999|      719991|
| O001201|      P007|       9|     79999|      719991|
| O003884|      P007|       9|     79999|      719991|
| O001300|      P007|       9|     79999|      719991|
| O003897|      P007|       9|     79999|      719991|
| O001796|

In [ ]:
df7=sales_df.filter(col("discount_pct")>10).withColumn(
    "discounted_sales",
    col("quantity") * col("unit_price") * (1 - col("discount_pct") / 100)
).orderBy(col("discounted_sales").desc())

df7.show(15)

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+----------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|discounted_sales|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+----------------+
| O008143|2026-01-17|     S03|      P007|       9|     79999|          15|          Cash|   Completed|       Online|       611992.35|
| O002940|2026-02-18|     S02|      P007|       9|     79999|          15|   Net Banking|   Completed|       Online|       611992.35|
| O006170|2026-03-30|     S04|      P007|       9|     79999|          15|          Card|   Completed|     In Store|       611992.35|
| O007875|2026-03-05|     S07|      P007|       9|     79999|          15|   Net Banking|   Cancelled|       Online|       611992.35|
| O003897|2026-05-14|     S02|      P007|       9|     79999| 

In [ ]:
df8=sales_df.groupBy("payment_method")\
  .agg(sum("quantity").alias("total_quantities")).sort(col("total_quantities").desc())
df8.show()

+--------------+----------------+
|payment_method|total_quantities|
+--------------+----------------+
|          Cash|           31754|
|          Card|           31327|
|           UPI|           30894|
|   Net Banking|           30889|
+--------------+----------------+



In [ ]:
df9=sales_df.groupBy("sales_channel","order_status")\
  .agg(count("*").alias("transaction_count"))
df9.show()

+-------------+------------+-----------------+
|sales_channel|order_status|transaction_count|
+-------------+------------+-----------------+
|       Online|   Completed|            10797|
|     In Store|   Cancelled|              592|
|       Online|    Returned|              987|
|       Online|   Cancelled|              633|
|     In Store|    Returned|             1013|
|     In Store|   Completed|            10978|
+-------------+------------+-----------------+



In [ ]:
df10=sales_df.select("order_id","store_id","product_id","quantity")
df10=df10.drop("store_id")
df10.first()

Row(order_id='O000001', product_id='P002', quantity=3)

**Part C — Joins and aggregations**

In [ ]:
df11=sales_df.join(products_df,on="product_id",how="inner")\
  .select("product_name","category","quantity","unit_price")
df11.show(20)

+-------------------+-----------+--------+----------+
|       product_name|   category|quantity|unit_price|
+-------------------+-----------+--------+----------+
|Mechanical Keyboard|Accessories|       3|      2999|
|      Smartphone 5G|    Mobiles|       6|     29999|
|     Android Tablet|    Tablets|       6|     22999|
|     Android Tablet|    Tablets|       7|     22999|
|          Laptop 16|  Computers|       2|     79999|
|          Webcam HD|Accessories|       8|      2499|
|  Bluetooth Speaker|      Audio|       8|      3999|
|Mechanical Keyboard|Accessories|       9|      2999|
|          Laptop 16|  Computers|       5|     79999|
|          Laptop 14|  Computers|       5|     55999|
|          Laptop 16|  Computers|       1|     79999|
|  Bluetooth Speaker|      Audio|       9|      3999|
|   External HDD 2TB|    Storage|       6|      5999|
|          Webcam HD|Accessories|       8|      2499|
|          Laptop 16|  Computers|       7|     79999|
|          Laptop 14|  Compu

In [ ]:
df12=sales_df.join(stores_df,"store_id").groupBy("city").agg(
    sum("quantity").alias("total_quantity")
)
df12.show()

+---------+--------------+
|     city|total_quantity|
+---------+--------------+
|  Chennai|         15727|
|   Mumbai|         15489|
|  Kolkata|         15619|
|     Pune|         15407|
|    Delhi|         15101|
|Bengaluru|         15718|
|Hyderabad|         15784|
|   Jaipur|         16019|
+---------+--------------+



In [ ]:
df13=sales_df.join(products_df,"product_id")\
  .join(stores_df,"store_id")\
  .filter(col("order_status")=="Completed")\
  .withColumn(
    "revenue",
    col("quantity") * col("unit_price"))\
  .groupBy("category","region")\
  .agg(sum("revenue").alias("total_revenue"))
df13.show()

+-----------+------+-------------+
|   category|region|total_revenue|
+-----------+------+-------------+
|   Displays| North|     32468090|
|   Displays| South|     45761308|
|    Storage| South|     36976319|
|   Displays|  East|     13616199|
|      Audio|  East|     12043135|
|    Tablets| North|     38270336|
|    Mobiles| South|    120674524|
|    Storage|  East|     12140136|
|    Mobiles| North|     75482543|
|    Mobiles|  East|     41178126|
|Accessories|  East|      9093361|
|      Audio|  West|     25317111|
|    Tablets|  West|     37488370|
|   Displays|  West|     28286336|
|      Audio| South|     34629665|
|      Audio| North|     24327211|
|    Storage|  West|     23673341|
|  Computers| North|    240756425|
|  Computers|  West|    221092729|
|    Mobiles|  West|     80372388|
+-----------+------+-------------+
only showing top 20 rows


In [ ]:
df14=sales_df.filter(col("order_status") == "Completed")\
  .join(products_df, on="product_id", how="inner")\
  .withColumn("discounted_revenue", col("quantity") * col("unit_price") * (1 - col("discount_pct") / 100))\
  .groupBy("product_id","product_name")\
  .agg(sum("discounted_revenue").alias("total_discount_revenue"))\
  .orderBy(col("total_discount_revenue").desc())

df14.show(5)


+----------+---------------+----------------------+
|product_id|   product_name|total_discount_revenue|
+----------+---------------+----------------------+
|      P007|      Laptop 16|  5.2272546585000014E8|
|      P006|      Laptop 14|   3.830051604999999E8|
|      P014|  Smartphone 5G|   2.060406317500002E8|
|      P008| Android Tablet|  1.5040656029999995E8|
|      P004|27-inch Monitor|  1.1406074015000013E8|
+----------+---------------+----------------------+
only showing top 5 rows


**Part D — Understanding lazy execution**

In [ ]:
df15 = sales_df.filter(col("quantity") > 5)
print("Filter completed")

df15 = df15.select(
    "product_id",
    "quantity",
    "unit_price"
)
print("Select completed")

df15 = df15.withColumn(
    "gross_amount",
    col("quantity") * col("unit_price")
)
print("WithColumn completed")

df15 = df15.groupBy("product_id").agg(
    sum("gross_amount").alias("total_amount")
)
print("GroupBy completed")

df15 = df15.orderBy(col("total_amount").desc())
print("OrderBy completed")

df15.show()

Filter completed
Select completed
WithColumn completed
GroupBy completed
OrderBy completed
+----------+------------+
|product_id|total_amount|
+----------+------------+
|      P007|   425994675|
|      P006|   297858681|
|      P014|   167304423|
|      P008|   119410808|
|      P004|    87663843|
|      P015|    76434540|
|      P011|    52239195|
|      P012|    41952006|
|      P013|    34392267|
|      P010|    23414145|
|      P002|    16152614|
|      P005|    14481705|
|      P009|    10802596|
|      P003|    10152054|
|      P001|     4676547|
+----------+------------+



In [ ]:
df15 = sales_df.filter(col("quantity") > 5)
print("Filter completed")

df15 = df15.select(
    "product_id",
    "quantity",
    "unit_price"
)
print("Select completed")

df15 = df15.withColumn(
    "gross_amount",
    col("quantity") * col("unit_price")
)
print("WithColumn completed")

df15 = df15.groupBy("product_id").agg(
    sum("gross_amount").alias("total_amount")
)
print("GroupBy completed")

df15 = df15.orderBy(col("total_amount").desc())
print("OrderBy completed")

df15.show()

Filter completed
Select completed
WithColumn completed
GroupBy completed
OrderBy completed
+----------+------------+
|product_id|total_amount|
+----------+------------+
|      P007|   425994675|
|      P006|   297858681|
|      P014|   167304423|
|      P008|   119410808|
|      P004|    87663843|
|      P015|    76434540|
|      P011|    52239195|
|      P012|    41952006|
|      P013|    34392267|
|      P010|    23414145|
|      P002|    16152614|
|      P005|    14481705|
|      P009|    10802596|
|      P003|    10152054|
|      P001|     4676547|
+----------+------------+



In [ ]:
df16 = sales_df.filter(
    col("quantity") > 5
)

df16.explain()

df16.show()

df16.count()

== Physical Plan ==
*(1) Filter (isnotnull(quantity#64) AND (quantity#64 > 5))
+- FileScan csv [order_id#60,order_date#61,store_id#62,product_id#63,quantity#64,unit_price#65,discount_pct#66,payment_method#67,order_status#68,sales_channel#69] Batched: false, DataFilters: [isnotnull(quantity#64), (quantity#64 > 5)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [IsNotNull(quantity), GreaterThan(quantity,5)], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...


+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000002|2026-05-29|     S07|      

11087

In [ ]:
df17 = sales_df.join(
    products_df,
    "product_id"
).groupBy(
    "category"
).agg(
    sum("quantity").alias("total_quantity")
)

result = df17.collect()

for row in result:
    print(row)

Row(category='Computers', total_quantity=16258)
Row(category='Displays', total_quantity=7870)
Row(category='Mobiles', total_quantity=16572)
Row(category='Accessories', total_quantity=41839)
Row(category='Storage', total_quantity=17251)
Row(category='Audio', total_quantity=17159)
Row(category='Tablets', total_quantity=7915)


collect() brings the result to the driver's memory. Collecting a small aggregated result containing only a few categories is safer than collecting all 25,000 rows, which could consume much more memory.

In [ ]:
df18=sales_df.filter(col("order_status")=="Completed")\
  .withColumn("discounted_revenue",col("quantity")*col("unit_price")*(1-col("discount_pct")/100))
df18.write.mode("overwrite").parquet("completed_orders.parquet")

The write operation is an action. It triggers Spark to execute the transformations and save the resulting DataFrame.